# WB Case — модель прогноза отгрузок (target_2h)

**Задача.** Предсказать `target_2h` — число ёмкостей, отгружаемых по маршруту за 2-часовое окно, для каждой точки тестового набора.

**Метрика соревнования.** `Score = WAPE + |Relative Bias|`, минимизируется.

**Данные.**
- Train: `data/train_team_track.parquet` — ~4.34M строк, period 01.03.2025 — 30.05.2025 10:30, 1000 маршрутов, 53 склада.
- Test: `data/test_team_track.parquet` — 10k строк, only `id / route_id / timestamp`, 30.05.2025 11:00 — 15:30 (сразу после train).

**План ноутбука.**
1. Загрузка данных и быстрый EDA.
2. Фичи: временные, агрегаты по маршруту/складу, лаги (векторно), агрегаты по `status_*`.
3. Time-split валидация (последние N дней train).
4. LightGBM c категориальными `route_id` / `office_from_id`.
5. Оценка WAPE + |RBias|, пост-калибровка под RBias.
6. Дообучение на полном train и submission.


## 1. Импорты и загрузка

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
SEED = 42


In [2]:
train = pd.read_parquet('data/train_team_track.parquet')
test  = pd.read_parquet('data/test_team_track.parquet')

# office_from_id в test отсутствует — восстанавливаем из train (привязка маршрут->склад фиксирована)
route_office = train[['route_id', 'office_from_id']].drop_duplicates().set_index('route_id')['office_from_id']
test['office_from_id'] = test['route_id'].map(route_office).astype(train['office_from_id'].dtype)

print('train:', train.shape, '| test:', test.shape)
print('train time:', train['timestamp'].min(), '->', train['timestamp'].max())
print('test  time:', test['timestamp'].min(),  '->', test['timestamp'].max())


train: (4342000, 12) | test: (10000, 4)
train time: 2025-03-01 00:00:00 -> 2025-05-30 10:30:00
test  time: 2025-05-30 11:00:00 -> 2025-05-30 15:30:00


## 2. Быстрый EDA

In [5]:
train

,office_from_id,route_id,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,target_2h
0,4,29,2025-03-01 00:00:00,3105,340,2160,484,4018,3462,0,0,27.0
1,4,29,2025-03-01 00:30:00,2813,388,2058,373,1363,1657,9380,0,27.0
2,4,29,2025-03-01 01:00:00,2465,293,2098,472,3195,3325,0,0,23.0
3,4,29,2025-03-01 01:30:00,1977,252,2351,310,3314,3243,0,0,37.0
4,4,29,2025-03-01 02:00:00,1585,206,2500,300,2130,1604,0,0,31.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4341995,53,526,2025-05-30 08:30:00,0,72,0,418,404,221,0,548,99.0
4341996,53,526,2025-05-30 09:00:00,0,60,0,275,551,686,783,525,91.0
4341997,53,526,2025-05-30 09:30:00,0,96,0,258,695,564,0,1150,89.0
4341998,53,526,2025-05-30 10:00:00,0,85,0,274,896,1547,0,1430,88.0


In [3]:
print(train['target_2h'].describe())
print('доля нулей:', (train['target_2h'] == 0).mean())
print('маршрутов:', train['route_id'].nunique(), '| складов:', train['office_from_id'].nunique())
train.head(3)


count    4.342000e+06
mean     6.874518e+01
std      6.748811e+01
min      0.000000e+00
25%      1.900000e+01
50%      4.800000e+01
75%      1.000000e+02
max      1.517000e+03
Name: target_2h, dtype: float64
доля нулей: 0.048664440350069095
маршрутов: 1000 | складов: 53


,office_from_id,route_id,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,status_7,status_8,target_2h
0,4,29,2025-03-01 00:00:00,3105,340,2160,484,4018,3462,0,0,27.0
1,4,29,2025-03-01 00:30:00,2813,388,2058,373,1363,1657,9380,0,27.0
2,4,29,2025-03-01 01:00:00,2465,293,2098,472,3195,3325,0,0,23.0


In [4]:
# Сколько точек на маршрут
pts_per_route = train.groupby('route_id').size()
print('точек на маршрут — min/median/max:', pts_per_route.min(), pts_per_route.median(), pts_per_route.max())

# Статусы: краткая статистика
status_cols = [f'status_{i}' for i in range(1, 9)]
train[status_cols].describe().T[['mean', 'std', 'max']]


точек на маршрут — min/median/max: 4342 4342.0 4342


,mean,std,max
status_1,1310.748731,2501.866251,36491.0
status_2,146.998607,206.666088,17150.0
status_3,1278.569003,2310.502482,26020.0
status_4,1102.996098,1740.374614,21170.0
status_5,1555.774074,2194.728055,26565.0
status_6,1555.737880,2255.212710,27702.0
status_7,2243.071239,4913.773645,76319.0
status_8,916.167541,1872.031485,52039.0


## 3. Временные фичи

Генерируются одинаково в train/test.

In [ ]:
def add_time_features(df):
    ts = df['timestamp']
    df['hour']       = ts.dt.hour.astype('int16')
    df['minute']     = ts.dt.minute.astype('int16')
    df['dow']        = ts.dt.dayofweek.astype('int16')
    df['day']        = ts.dt.day.astype('int16')
    df['week']       = ts.dt.isocalendar().week.astype('int16')
    df['is_weekend'] = (df['dow'] >= 5).astype('int8')
    df['slot']       = (df['hour'] * 2 + df['minute'] // 30).astype('int16')
    return df

train = add_time_features(train)
test  = add_time_features(test)


## 4. Train/validation split

Берём последние 3 дня train в качестве валидации — это надёжнее, чем 4.5 часа. Тест идёт сразу после train (4.5 ч), но валидация на нескольких днях даёт более устойчивую оценку.

In [ ]:
VAL_DAYS = 3
cutoff = train['timestamp'].max().normalize() - pd.Timedelta(days=VAL_DAYS - 1)
print('val cutoff:', cutoff)

val_mask = train['timestamp'] >= cutoff
tr_df  = train[~val_mask].reset_index(drop=True)
val_df = train[ val_mask].reset_index(drop=True)
print('tr:', tr_df.shape, '| val:', val_df.shape)


## 5. Агрегаты из train-части (без утечки)

Считаем средние/медианы `target_2h` и `status_*` по разным ключам **только на tr_df**, чтобы val и test не видели будущее.

In [ ]:
def build_aggregates(df):
    aggs = {}
    aggs['route'] = df.groupby('route_id')['target_2h'].agg(
        route_mean='mean', route_std='std', route_median='median')
    aggs['route_hour'] = df.groupby(['route_id', 'hour'])['target_2h'].agg(
        route_hour_mean='mean', route_hour_median='median')
    aggs['route_dow'] = df.groupby(['route_id', 'dow'])['target_2h'].agg(
        route_dow_mean='mean', route_dow_median='median')
    aggs['route_dow_hour'] = df.groupby(['route_id', 'dow', 'hour'])['target_2h'].agg(
        route_dow_hour_mean='mean')
    aggs['office'] = df.groupby('office_from_id')['target_2h'].agg(
        office_mean='mean', office_std='std')
    aggs['office_hour'] = df.groupby(['office_from_id', 'hour'])['target_2h'].agg(
        office_hour_mean='mean')

    # Агрегаты по статусам (доступны только в train, поэтому переносим как исторические фичи по маршруту)
    status_cols = [f'status_{i}' for i in range(1, 9)]
    aggs['route_status'] = df.groupby('route_id')[status_cols].mean().add_suffix('_route_mean')
    return aggs

def apply_aggregates(df, aggs):
    for key, tbl in aggs.items():
        df = df.merge(tbl, left_on=list(tbl.index.names), right_index=True, how='left')
    return df

aggs_tr = build_aggregates(tr_df)
print('агрегатов построено:', list(aggs_tr.keys()))


## 6. Лаговые фичи (векторно)

Исходный скрипт использовал `iterrows` для лагов — это медленно на 4.3M строк. Здесь считаем лаги через `groupby().shift()`.

Важный момент: для val/test лаги берутся из непрерывной истории, которая включает все предыдущие точки train. Соберём единый `full` DataFrame, упорядочим по (route_id, timestamp) и посчитаем лаги один раз.

In [ ]:
LAGS = [1, 2, 3, 4, 5, 6, 7, 8]

def add_lag_features(hist_df, target_df, lags=LAGS):
    '''hist_df: строки с target_2h (train или tr). target_df: строки, для которых считаем лаги.
    Возвращает target_df с колонками lag_i, lag_mean_4, lag_mean_8.'''
    hist = hist_df[['route_id', 'timestamp', 'target_2h']].copy()
    tgt  = target_df[['route_id', 'timestamp']].copy()
    tgt['target_2h'] = np.nan

    full = pd.concat([hist, tgt], ignore_index=True)
    full = full.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

    grp = full.groupby('route_id')['target_2h']
    for i in lags:
        full[f'lag_{i}'] = grp.shift(i)
    full['lag_mean_4'] = full[[f'lag_{i}' for i in range(1, 5)]].mean(axis=1)
    full['lag_mean_8'] = full[[f'lag_{i}' for i in range(1, 9)]].mean(axis=1)

    lag_cols = [f'lag_{i}' for i in lags] + ['lag_mean_4', 'lag_mean_8']
    merged = target_df.merge(
        full[['route_id', 'timestamp'] + lag_cols],
        on=['route_id', 'timestamp'], how='left')
    return merged


In [ ]:
# Для валидации: история = tr_df, цели = val_df
val_feat = add_lag_features(tr_df, val_df)
val_feat = apply_aggregates(val_feat, aggs_tr)

# Для train-части: лаги внутри самого tr_df (shift по route_id)
tr_feat = tr_df.sort_values(['route_id', 'timestamp']).reset_index(drop=True)
grp = tr_feat.groupby('route_id')['target_2h']
for i in LAGS:
    tr_feat[f'lag_{i}'] = grp.shift(i)
tr_feat['lag_mean_4'] = tr_feat[[f'lag_{i}' for i in range(1, 5)]].mean(axis=1)
tr_feat['lag_mean_8'] = tr_feat[[f'lag_{i}' for i in range(1, 9)]].mean(axis=1)
tr_feat = apply_aggregates(tr_feat, aggs_tr)
tr_feat = tr_feat.dropna(subset=['lag_8']).reset_index(drop=True)

print('tr_feat:', tr_feat.shape, '| val_feat:', val_feat.shape)


## 7. Обучение LightGBM

In [ ]:
FEATURES = [
    # время
    'hour', 'minute', 'dow', 'day', 'week', 'is_weekend', 'slot',
    # идентификаторы (категориальные)
    'office_from_id', 'route_id',
    # агрегаты по маршруту
    'route_mean', 'route_std', 'route_median',
    'route_hour_mean', 'route_hour_median',
    'route_dow_mean', 'route_dow_median',
    'route_dow_hour_mean',
    # агрегаты по складу
    'office_mean', 'office_std', 'office_hour_mean',
    # лаги
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8',
    'lag_mean_4', 'lag_mean_8',
    # статусы — исторические средние по маршруту
] + [f'status_{i}_route_mean' for i in range(1, 9)]

CAT_FEATURES = ['office_from_id', 'route_id', 'hour', 'dow', 'slot', 'is_weekend']

X_tr,  y_tr  = tr_feat[FEATURES],  tr_feat['target_2h']
X_val, y_val = val_feat[FEATURES], val_feat['target_2h']
print('X_tr:', X_tr.shape, '| X_val:', X_val.shape)


In [ ]:
params = {
    'objective': 'regression_l1',   # MAE хорошо коррелирует с WAPE
    'metric': 'mae',
    'learning_rate': 0.05,
    'num_leaves': 191,
    'min_child_samples': 100,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'lambda_l2': 1.0,
    'seed': SEED,
    'verbose': -1,
    'n_jobs': -1,
}

dtrain = lgb.Dataset(X_tr,  label=y_tr,  categorical_feature=CAT_FEATURES)
dval   = lgb.Dataset(X_val, label=y_val, categorical_feature=CAT_FEATURES, reference=dtrain)

model = lgb.train(
    params, dtrain,
    num_boost_round=3000,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(75), lgb.log_evaluation(100)],
)
print('best_iteration:', model.best_iteration)


## 8. Оценка и пост-калибровка RBias

In [ ]:
def wape_rbias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    wape  = np.abs(y_pred - y_true).sum() / y_true.sum()
    rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
    return wape, rbias, wape + rbias

val_pred = np.clip(model.predict(X_val), 0, None)
wape, rbias, score = wape_rbias(y_val, val_pred)
print(f'до калибровки:  WAPE={wape:.4f}  |RBias|={rbias:.4f}  Score={score:.4f}')

# Пост-калибровка: умножаем предсказания на соотношение сумм, чтобы обнулить RBias на val
scale = y_val.sum() / val_pred.sum()
val_pred_cal = np.clip(val_pred * scale, 0, None)
wape_c, rbias_c, score_c = wape_rbias(y_val, val_pred_cal)
print(f'после калибровки: WAPE={wape_c:.4f}  |RBias|={rbias_c:.4f}  Score={score_c:.4f}  scale={scale:.4f}')


In [ ]:
# Важность фич
imp = pd.Series(model.feature_importance(importance_type='gain'), index=FEATURES)
imp.sort_values(ascending=False).head(20)


## 9. Финальная модель + предсказание test

Пересчитываем агрегаты на **полном** train, перестраиваем фичи и обучаем на всём train (на `best_iteration`, без раннего останова).

In [ ]:
aggs_full = build_aggregates(train)

# full train features
full = train.sort_values(['route_id', 'timestamp']).reset_index(drop=True)
grp = full.groupby('route_id')['target_2h']
for i in LAGS:
    full[f'lag_{i}'] = grp.shift(i)
full['lag_mean_4'] = full[[f'lag_{i}' for i in range(1, 5)]].mean(axis=1)
full['lag_mean_8'] = full[[f'lag_{i}' for i in range(1, 9)]].mean(axis=1)
full = apply_aggregates(full, aggs_full)
full = full.dropna(subset=['lag_8']).reset_index(drop=True)

X_full, y_full = full[FEATURES], full['target_2h']
print('X_full:', X_full.shape)


In [ ]:
dfull = lgb.Dataset(X_full, label=y_full, categorical_feature=CAT_FEATURES)
final_model = lgb.train(
    params, dfull,
    num_boost_round=int(model.best_iteration * 1.05),  # небольшой запас
)


In [ ]:
# test features
test_feat = add_lag_features(train, test)
test_feat = apply_aggregates(test_feat, aggs_full)

X_test = test_feat[FEATURES]
print('X_test:', X_test.shape, '| NaN в лагах:', X_test[[f'lag_{i}' for i in LAGS]].isna().any().any())


In [ ]:
test_pred = np.clip(final_model.predict(X_test), 0, None)

# Применяем ту же калибровку, что оценили на валидации
test_pred_cal = np.clip(test_pred * scale, 0, None)

print('raw : mean=%.2f std=%.2f' % (test_pred.mean(), test_pred.std()))
print('cal : mean=%.2f std=%.2f' % (test_pred_cal.mean(), test_pred_cal.std()))


In [ ]:
submission = pd.DataFrame({'id': test['id'], 'y_pred': test_pred_cal})
submission.to_csv('submission.csv', index=False)
print('submission saved:', submission.shape)
submission.head()


## Что ещё можно попробовать

- **TimeSeriesSplit** на 3-5 фолдов вместо одного holdout — устойчивее к выбросу дня.
- **Кастомная loss** под WAPE (веса `1 / y.sum()`) или Tweedie — для правой хвостатости.
- **Two-stage** (нулевые/ненулевые отгрузки): классификатор + регрессор — учитывая ~5% нулей.
- **Status-фичи как временные ряды** — не только среднее по маршруту, но и (route_id, hour)-средние статусов за последние часы.
- **Тюнинг** num_leaves / min_child_samples по фолдам, blending нескольких сидов.
- **Округление** до ближайшего целого — если ёмкости целочисленны и это улучшает WAPE.
